# Cypher

A continuación, se muestra los queries utilizados para obtener cada uno de los resultados mostrados en el documento final.

## 1. Arquitectura real del grafo

```cypher
MATCH (b:Buyer)-[i:INITIATED]->(p:Procedure)
MATCH (p)-[a:AWARDED_TO]->(s:Supplier)
OPTIONAL MATCH (p)-[cl:CLASSIFIED_AS]->(c:CPC)
WHERE b.dataset = 'SIE_2025'
RETURN b, i, p, a, s, cl, c
LIMIT 20
```

## 2. Relaciones recurrentes Buyer–Supplier–CPC5 en 90 días


```cypher
MATCH (b:Buyer)-[:INITIATED]->(p:Procedure)-[a:AWARDED_TO]->(s:Supplier)
MATCH (p)-[:CLASSIFIED_AS]->(c:CPC)
WHERE b.dataset = 'SIE_2025'
  AND a.award_date IS NOT NULL

WITH
    b,
    s,
    c,
    p,
    a,
    date(a.award_date) AS award_date

WITH
    b,
    s,
    c,
    collect({
        procedure: p,
        award: a,
        award_date: award_date
    }) AS observations

UNWIND observations AS start_obs

WITH
    b,
    s,
    c,
    start_obs,
    [
        x IN observations
        WHERE x.award_date >= start_obs.award_date
          AND x.award_date <= start_obs.award_date + duration({days: 90})
    ] AS window_90d

WHERE size(window_90d) >= 4

WITH b, s, c, window_90d
ORDER BY size(window_90d) DESC
LIMIT 1

UNWIND window_90d AS obs

RETURN
    b,
    obs.procedure AS p,
    obs.award AS a,
    s,
    c
````

## 3. Actor con mayor centralidad de intermediación — Betweenness

```cypher
MATCH (n)
WHERE (n:Buyer OR n:Supplier)
  AND n.betweenness IS NOT NULL

WITH n
ORDER BY n.betweenness DESC
LIMIT 1

OPTIONAL MATCH (b:Buyer)-[r:CONTRACTS_WITH]->(s:Supplier)
WHERE b = n OR s = n

RETURN b, r, s
ORDER BY r.total_amount DESC
LIMIT 30
```

## 4. Trazabilidad desde una relación Buyer–Supplier hasta sus procedimientos

```cypher
CALL gds.betweenness.stream('sie_buyer_supplier_2025')
YIELD nodeId, score

WITH
    gds.util.asNode(nodeId) AS actor,
    score

WHERE actor:Buyer OR actor:Supplier

WITH actor, score
ORDER BY score DESC
LIMIT 1

MATCH (b:Buyer)-[r:CONTRACTS_WITH]->(s:Supplier)
WHERE b = actor OR s = actor

RETURN
    b,
    r,
    s

ORDER BY r.total_amount DESC
LIMIT 30
```